# Tanshi — Production Avatar Training Notebook (AI Creator Platform)

The complete, resumable production training environment for the Tanshi
digital avatar (frozen dataset: 461 accepted clips · 61 min · readiness 95/100).

**Supported models:** `musetalk` · `latentsync` · `echomimic` · `hallo2`

**Structure** (Run All is safe — training NEVER auto-starts):

| cells | what happens |
|---|---|
| CONFIG | unified configuration — the only cell you edit |
| 1 · 1b | mount Drive · **optional dev environment** (INFRA 2.0 SSH/tunnel/tmux/Claude Code — off unless `ENABLE_SSH = True`) |
| 2–6 | GPU check → clone repo → locate + sha256-verify dataset → stage to local disk |
| 7 | install the selected model (Colab-safe: keeps Colab's torch, relaxes dead pins) |
| 8–10 | checkpoint manager + training logger · training launcher (defined, not run) · benchmark folder structure |
| 11–14 | pre-training validation gate · pretrained weight manager (Drive-cached, downloads once) · dataset summary · **training readiness report** |
| 15 | **START TRAINING** — runs only when you set `CONFIRM_START = True` in that cell |

Run All ends at **READY TO START TRAINING** and waits for you.
Checkpoints, logs, metrics, weights, benchmark outputs, and infra logs all
live on Google Drive (`avatar_training/`), so a disconnected Colab session
resumes where it stopped — rerun the notebook and the latest checkpoint is
picked up automatically. For long runs, enable the dev environment and launch
training inside the `avatar-training` tmux session over SSH.

In [ ]:
# ============================ CONFIG — the only cell you edit (SECTION 3) ============================
MODEL_NAME = "musetalk"       #@param ["musetalk", "latentsync", "echomimic", "hallo2"]

# --- repository -----------------------------------------------------------------
REPO_URL = "https://github.com/nahatadhananjay33-svg/ai-creator-platform.git"
BRANCH   = "main"

# --- dataset (frozen — never modified by this notebook) ---------------------------
DATASET_DIRNAME = "avatar_dataset"
# uploaded 2026-07-17; clear to force auto-discovery under MyDrive
DATASET_PATH    = "/content/drive/MyDrive/Ai_creator/Digital_Avatar_Tanshi/avatar_dataset"
FULL_HASH_VERIFY = False              # True = re-hash all 16.7 GB on Drive
STAGE_TO_LOCAL   = True               # copy accepted clips to /content for fast I/O

# --- Drive output layout (everything persistent lives here) -----------------------
WORK_DIRNAME    = "avatar_training"
OUTPUT_PATH     = f"/content/drive/MyDrive/{WORK_DIRNAME}"
CHECKPOINT_PATH = f"{OUTPUT_PATH}/checkpoints/{MODEL_NAME}"
LOG_PATH        = f"{OUTPUT_PATH}/logs/{MODEL_NAME}"
METRICS_PATH    = f"{OUTPUT_PATH}/metrics/{MODEL_NAME}"
WEIGHTS_PATH    = f"{OUTPUT_PATH}/weights/{MODEL_NAME}"
BENCHMARK_PATH  = f"{OUTPUT_PATH}/benchmarks/{MODEL_NAME}"

# --- developer environment (OPTIONAL — INFRA 2.0; see the cell after Drive mount) --
ENABLE_SSH                 = False    # True = run the dev bootstrap (SSH + tunnel)
SSH_MODE                   = "quick"  # quick = trycloudflare ephemeral tunnel
AUTO_START_TMUX            = True     # create/reuse tmux session 'avatar-training'
AUTO_START_WATCHDOG        = True     # keep tunnel + sshd alive across hiccups
ENABLE_CHECKPOINT_COMMANDS = True     # install checkpoint / recover / dev commands
ENABLE_CLAUDE_CODE         = True     # install Claude Code CLI if missing

# --- training hyperparameters ------------------------------------------------------
# Mapped to each repo's flags via the launcher's per-model template; flag names
# follow the upstream READMEs — if a repo rejects one, set USE_HPARAM_TEMPLATE
# = False (cell 8) and pass exact flags via EXTRA_ARGS instead.
BATCH_SIZE          = 4
EPOCHS              = 20
LEARNING_RATE       = 1e-5
NUM_WORKERS         = 2
PRECISION           = "fp16"          # fp16 | bf16 | fp32 (bf16 needs A100/L4)
RESUME              = True            # resume from the latest Drive checkpoint
SAVE_INTERVAL       = 500             # steps between checkpoint saves
VALIDATION_INTERVAL = 500             # steps between validation passes
SEED                = 20260718
EXTRA_ARGS          = ""              # appended verbatim to the train command

MODEL = MODEL_NAME                    # alias used by the setup cells below
print(f"model={MODEL_NAME}  branch={BRANCH}  precision={PRECISION}  "
      f"resume={RESUME}  ssh={'on' if ENABLE_SSH else 'off'}  "
      f"outputs -> {OUTPUT_PATH}")

In [ ]:
# ============================ 1. Mount Google Drive ============================
from pathlib import Path
from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
MYDRIVE = Path("/content/drive/MyDrive")
WORK = Path(OUTPUT_PATH)
for p in (CHECKPOINT_PATH, LOG_PATH, METRICS_PATH, WEIGHTS_PATH, BENCHMARK_PATH):
    Path(p).mkdir(parents=True, exist_ok=True)
print(f"Drive mounted. Persistent outputs: {WORK}")

## Developer Environment (Optional)

Off by default (`ENABLE_SSH = False` in CONFIG) — Run All skips straight to
training setup.

Turn it on to reuse the platform's **INFRA 2.0 Colab Dev Bootstrap** (the
canonical implementation from `tanshi_voice_cloning_setup.ipynb`, fetched and
executed verbatim — no duplicated code): SSH over a Cloudflare tunnel, a
watchdog that survives disconnects, Claude Code, git identity, and the
`checkpoint` / `recover` / `dev` commands. Extras added for training runs:

- a `tmux` session **`avatar-training`** (created/reused when
  `AUTO_START_TMUX = True`) — run long trainings inside it over SSH so they
  outlive dropped connections;
- infra logs (tunnel hostname, cloudflared, watchdog) sync to
  `avatar_training/logs/<model>/infra/` on Drive every 5 minutes;
- `checkpoint "msg"` pushes repo work off-box during training; `recover`
  restores after a runtime reclaim.

Training never starts here — the notebook still ends at
**READY TO START TRAINING**.

In [ ]:
# ============================ 1b. Developer environment (OPTIONAL — INFRA 2.0, reused) ============================
if not ENABLE_SSH:
    print("ENABLE_SSH = False - developer bootstrap skipped; continuing to training setup.")
else:
    import json as _json
    import os as _os
    import shutil as _sh
    import subprocess as _sp
    import urllib.request as _rq

    # toggles exported for the bootstrap (it is idempotent and self-healing)
    _os.environ.update({
        "SSH_MODE": SSH_MODE,
        "AUTO_START_WATCHDOG": str(AUTO_START_WATCHDOG),
        "ENABLE_CHECKPOINT_COMMANDS": str(ENABLE_CHECKPOINT_COMMANDS),
        "ENABLE_CLAUDE_CODE": str(ENABLE_CLAUDE_CODE),
    })

    # REUSE the canonical INFRA 2.0 bootstrap — fetched verbatim from the voice
    # notebook (single source of truth; nothing is duplicated here). It installs
    # packages, configures SSH + the Cloudflare tunnel, starts sshd + watchdog,
    # installs Claude Code, configures git and the checkpoint/recover/dev
    # commands, and prints connection instructions.
    _url = (f"https://raw.githubusercontent.com/nahatadhananjay33-svg/"
            f"ai-creator-platform/{BRANCH}/notebooks/tanshi_voice_cloning_setup.ipynb")
    _vnb = _json.loads(_rq.urlopen(_url).read())
    _src = next("".join(c["source"]) for c in _vnb["cells"]
                if c["cell_type"] == "code"
                and "Colab Dev Bootstrap" in "".join(c["source"]))
    print("running the canonical INFRA 2.0 bootstrap "
          "(reused from tanshi_voice_cloning_setup.ipynb)...\n")
    exec(compile(_src, "colab_dev_bootstrap", "exec"))

    # --- avatar-specific: tmux session for long training runs ----------------------
    if AUTO_START_TMUX and _sh.which("tmux"):
        if _sp.run(["tmux", "has-session", "-t", "avatar-training"],
                   capture_output=True).returncode != 0:
            _sp.run(["tmux", "new-session", "-d", "-s", "avatar-training"])
            print("tmux: created session 'avatar-training'")
        else:
            print("tmux: session 'avatar-training' already running")
        print("      attach over SSH with: tmux attach -t avatar-training")

    # --- persist infra logs to Drive (tunnel/watchdog/recovery, every 5 min) -------
    _infra_logs = Path(LOG_PATH) / "infra"
    _infra_logs.mkdir(parents=True, exist_ok=True)
    _sp.Popen("nohup bash -c 'while true; do "
              f"cp -u /content/cloudflared.log \"{_infra_logs}/\" 2>/dev/null; "
              f"cp -u /content/tunnel_hostname.txt \"{_infra_logs}/\" 2>/dev/null; "
              f"cp -u /content/*watchdog*.log \"{_infra_logs}/\" 2>/dev/null; "
              "sleep 300; done' >/dev/null 2>&1 &", shell=True)
    print(f"infra logs sync to {_infra_logs} every 5 min")

    # --- validation -----------------------------------------------------------------
    def _stat(name, ok, extra=""):
        print(f"  {'[OK]  ' if ok else '[--]  '}{name}" + (f": {extra}" if extra else ""))

    _ps = _sp.run(["ps", "-eo", "args"], capture_output=True, text=True).stdout
    _hostfile = Path(globals().get("HOSTFILE", "/content/tunnel_hostname.txt"))
    _host = _hostfile.read_text().strip() if _hostfile.exists() else "no hostname yet"
    _git_id = _sp.run(["git", "config", "--global", "user.email"],
                      capture_output=True, text=True).stdout.strip()
    import torch as _tt

    print("\nDeveloper environment status:")
    _stat("SSH (sshd)", "sshd" in _ps)
    _stat("Tunnel (cloudflared)", "cloudflared" in _ps, _host)
    _stat("Claude Code", _sh.which("claude") is not None)
    _stat("Watchdog", "watchdog" in _ps.lower())
    _stat("Git identity", bool(_git_id), _git_id)
    _stat("GPU", _sh.which("nvidia-smi") is not None)
    _stat("CUDA", _tt.cuda.is_available())
    for _c in ("checkpoint", "recover", "dev"):
        _stat(f"'{_c}' command", _sh.which(_c) is not None)
    print("\nDev bootstrap done - training setup continues below "
          "(training still starts ONLY from the final cell).")

In [ ]:
# ============================ 2. GPU / CUDA detection ============================
import shutil, subprocess, sys

if not shutil.which("nvidia-smi"):
    raise SystemExit("No GPU runtime! Runtime -> Change runtime type -> GPU, then rerun.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)
import torch
print(f"torch {torch.__version__}  cuda available: {torch.cuda.is_available()}"
      f"  device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'n/a'}")
assert torch.cuda.is_available(), "CUDA not available - check the runtime type"

In [ ]:
# ============================ 3. Clone repo + base dependencies ============================
import os, subprocess
from pathlib import Path

REPO = Path("/content/ai_creator_platform")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)],
                   check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                # numpy<2 + numpy-1.x-built OpenCV: MuseTalk needs numpy<2; leaving
                # numpy unpinned lets it upgrade to 2.x under a numpy-1.x cv2 wheel ->
                # '_ARRAY_API not found' on import cv2. Keep the pair ABI-matched.
                "numpy<2", "opencv-python-headless==4.10.0.84",
                "openpyxl", "tqdm"], check=True)
# numpy HEALTH CHECK in a fresh interpreter: a broken hybrid install
# (leftover mixed 1.x/2.x binaries after upgrade/downgrade churn) imports
# fine in this kernel but crashes any fresh import with
# 'numpy.dtype size changed'. Repair = clean uninstall + reinstall.
_chk = subprocess.run([sys.executable, "-c", "import numpy.random"],
                      capture_output=True, text=True)
if _chk.returncode != 0:
    print("numpy on disk is BROKEN (mixed binaries) - clean reinstalling...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "numpy"])
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy<2"],
                   check=True)
    _chk2 = subprocess.run([sys.executable, "-c", "import numpy.random"],
                           capture_output=True, text=True)
    if _chk2.returncode != 0:
        raise SystemExit("numpy still broken after reinstall - use a fresh "
                         "runtime (Runtime -> Disconnect and delete runtime).")
    raise SystemExit("numpy repaired on disk. Now: Runtime -> Restart session, "
                     "then Run All again (staged data is kept).")

print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# ============================ 4. Locate the dataset on Drive ============================
DATASET_FOLDER_ID = "1JGyi0atGEA79-2v2l3SMSFNHMuG8Fw7f"   # Drive id of avatar_dataset

def find_dataset(mydrive: Path, dirname: str) -> Path:
    candidates = []
    if DATASET_PATH:
        candidates.append(Path(DATASET_PATH))
    # a My Drive SHORTCUT to a shared folder surfaces here in the Colab mount
    candidates.append(Path("/content/drive/.shortcut-targets-by-id")
                      / DATASET_FOLDER_ID / dirname)
    for p in candidates:
        if (p / "dataset.sqlite").exists():
            return p
    print("note: direct paths missing - walking MyDrive (can take a minute)...")
    hits = [Path(r) for r, dirs, files in os.walk(mydrive)
            if Path(r).name == dirname and "dataset.sqlite" in files]
    if len(hits) > 1:
        print(f"note: {len(hits)} candidates, using the first: {hits[0]}")
    if hits:
        return hits[0]
    raise SystemExit(
        f"'{dirname}' not reachable in this mount.\n"
        "The dataset lives in nahatadhananjay33@gmail.com's My Drive. If Drive shows\n"
        "it under 'Shared with me', you mounted a DIFFERENT account. Fix either way:\n"
        "  a) Runtime -> Disconnect and delete runtime, rerun, and when the Drive\n"
        "     popup asks which account to authorize, pick nahatadhananjay33@gmail.com; or\n"
        "  b) in Drive (this account), right-click avatar_dataset under Shared with me\n"
        "     -> Organise -> Add shortcut -> My Drive, then rerun this cell.")

DATASET = find_dataset(MYDRIVE, DATASET_DIRNAME)
print(f"Dataset: {DATASET}")

In [ ]:
# ============================ 5. Verify upload integrity (sha256 manifest) ============================
from production.cloud_setup.manifest import load_manifest, verify_against

man = load_manifest("production/cloud_setup/manifest/avatar_dataset_manifest.json")
print(f"manifest: {man['file_count']} files, {man['total_gb']} GB, "
      f"scope: {man['upload_scope']}")
res = verify_against(man, DATASET, quick=not FULL_HASH_VERIFY)
print(f"verified {res['verified']}/{res['expected_files']}  "
      f"missing {res['missing_count']}  size-bad {res['size_mismatch_count']}  "
      f"hash-bad {res['hash_mismatch_count']}")
assert res["ok"], f"UPLOAD INCOMPLETE/CORRUPT - first problems: " \
                  f"{(res['missing'] + res['size_mismatch'] + res['hash_mismatch'])[:5]}"
print("Upload integrity: PASS")

In [ ]:
# ============================ 6. Stage dataset to fast local disk ============================
# Drive's FUSE mount drops under sustained bulk reads (OSError errno 107,
# "Transport endpoint is not connected") - every copy retries with a forced
# remount, the same pattern the voice pipeline uses. Resume-safe: rerunning
# skips clips already staged at the right size.
import shutil, time

def _remount():
    try:
        from google.colab import drive as _drive
        _drive.mount("/content/drive", force_remount=True)
    except Exception as e:
        print(f"  remount attempt failed: {e}")

def robust_copy(src: Path, dst: Path, retries: int = 4) -> None:
    for attempt in range(1, retries + 1):
        try:
            shutil.copyfile(src, dst)
            return
        except OSError as e:
            if attempt == retries:
                raise
            print(f"  Drive hiccup (errno {e.errno}) on {src.name} - "
                  f"remounting, retry {attempt}/{retries - 1}")
            _remount()
            time.sleep(2.0 * attempt)

def robust_list(folder: Path, retries: int = 4):
    for attempt in range(1, retries + 1):
        try:
            return sorted(folder.glob("*"))
        except OSError:
            if attempt == retries:
                raise
            _remount()
            time.sleep(2.0 * attempt)

LOCAL_DATASET = Path("/content/avatar_dataset")
if STAGE_TO_LOCAL:
    (LOCAL_DATASET / "accepted").mkdir(parents=True, exist_ok=True)
    for name in ("dataset.sqlite", "dataset.csv", "dataset.xlsx"):
        if not (LOCAL_DATASET / name).exists():
            robust_copy(DATASET / name, LOCAL_DATASET / name)
    from tqdm.auto import tqdm
    clips = robust_list(DATASET / "accepted")
    for src in tqdm(clips, desc="staging", unit="clip"):
        dst = LOCAL_DATASET / "accepted" / src.name
        if not (dst.exists() and dst.stat().st_size == src.stat().st_size):  # resume
            robust_copy(src, dst)
    TRAIN_DATA = LOCAL_DATASET
else:
    TRAIN_DATA = DATASET
print(f"Training data root: {TRAIN_DATA} "
      f"({len(list((TRAIN_DATA / 'accepted').glob('*')))} clips)")

In [ ]:
# ============================ 7. Model registry + install selected model ============================
# Colab-safe install strategy: NEVER let a repo's requirements.txt replace
# Colab's own CUDA torch stack. Framework lines are filtered out; the rest
# installs in bulk; on failure each package retries individually, and a
# package whose exact PIN has no build for Colab's Python retries UNPINNED
# (latest compatible) - relaxed pins are listed at the end for review.
import importlib.metadata as _ilmd
import re
import subprocess
import sys
from pathlib import Path

# numpy version to preserve, captured ONCE per session before installs
# (cell 3 pins numpy<2 + a matched OpenCV; nothing may move them later)
_NP_ORIG = os.environ.setdefault("COLAB_NUMPY_ORIG", _ilmd.version("numpy"))

MODELS = {
    "musetalk": {
        "repo": "https://github.com/TMElyralab/MuseTalk.git",
        "install": ["REQUIREMENTS", "pip install --no-cache-dir -U openmim",
                     "mim install mmengine", "mim install 'mmcv>=2.0.1'",
                     "mim install 'mmdet>=3.1.0'", "mim install 'mmpose>=1.1.0'"],
        "weights": "sh ./download_weights.sh",
        "train": "python train.py --data_root {data} --output_dir {ckpt} {resume_flag}",
        "resume_flag": "--resume {latest}",
    },
    "latentsync": {
        "repo": "https://github.com/bytedance/LatentSync.git",
        "install": ["REQUIREMENTS"],
        "weights": "huggingface-cli download ByteDance/LatentSync-1.5 --local-dir checkpoints",
        "train": "python -m scripts.train_unet --config configs/unet/stage2.yaml "
                  "--data_dir {data} --output_dir {ckpt} {resume_flag}",
        "resume_flag": "--resume_from_checkpoint {latest}",
    },
    "echomimic": {
        "repo": "https://github.com/antgroup/echomimic.git",
        "install": ["REQUIREMENTS"],
        "weights": "git lfs install && git clone https://huggingface.co/BadToBest/EchoMimic pretrained_weights",
        "train": "accelerate launch train_stage1.py --config configs/train/stage1.yaml "
                  "--data_root {data} --output_dir {ckpt} {resume_flag}",
        "resume_flag": "--resume {latest}",
    },
    "hallo2": {
        "repo": "https://github.com/fudan-generative-vision/hallo2.git",
        "install": ["REQUIREMENTS", "pip install -e ."],
        "weights": "huggingface-cli download fudan-generative-ai/hallo2 --local-dir pretrained_models",
        "train": "accelerate launch scripts/train_stage1.py --config configs/train/stage1.yaml "
                  "--data_root {data} --output_dir {ckpt} {resume_flag}",
        "resume_flag": "--resume {latest}",
    },
}

# frameworks Colab already provides with matched CUDA builds - never reinstall
_SKIP = re.compile(r"^(torch|torchvision|torchaudio|tensorflow|tensorboard|jax|"
                   r"nvidia-|xformers|triton|numpy|scipy|opencv-python|"
                   r"opencv-contrib-python)\s*($|[=<>!~\[])", re.I)


def _pip(*args) -> bool:
    return subprocess.run([sys.executable, "-m", "pip", "install", *args]).returncode == 0


def install_requirements(repo_dir: Path):
    """Returns (relaxed_pins, failed). Skips Colab-provided frameworks."""
    req = repo_dir / "requirements.txt"
    if not req.exists():
        print("  (no requirements.txt)")
        return [], []
    keep, skipped = [], []
    for line in req.read_text().splitlines():
        s = line.split("#", 1)[0].strip()
        if not s:
            continue
        (skipped if _SKIP.match(s) else keep).append(s)
    if skipped:
        print(f"  keeping Colab's own versions of: {', '.join(skipped)}")
    colab_req = repo_dir / "requirements_colab.txt"
    colab_req.write_text("\n".join(keep))
    if _pip("-r", str(colab_req)):
        return [], []
    print("\n  bulk install failed - per-package pass (bad pins retry unpinned)...")
    relaxed, failed = [], []
    for pkg in keep:
        if _pip(pkg):
            continue
        name_only = re.split(r"[=<>!~;@]", pkg, 1)[0].strip()
        if name_only and name_only != pkg and _pip(name_only):
            relaxed.append(f"{pkg} -> {name_only} (latest)")
            print(f"  RELAXED: {pkg} -> latest {name_only}")
        else:
            failed.append(pkg)
            print(f"  FAILED : {pkg}")
    return relaxed, failed


cfg = MODELS[MODEL]
MODEL_DIR = Path(f"/content/{MODEL}")
if not MODEL_DIR.exists():
    subprocess.run(["git", "clone", cfg["repo"], str(MODEL_DIR)], check=True)
os.chdir(MODEL_DIR)

relaxed_pins, failed_steps = [], []
for step in cfg["install"]:
    if step == "REQUIREMENTS":
        r, f = install_requirements(MODEL_DIR)
        relaxed_pins += r
        failed_steps += [f"pip: {p}" for p in f]
    else:
        print(f"$ {step}")
        if subprocess.run(step, shell=True).returncode != 0:
            failed_steps.append(step)

# numpy consistency guard: if ANY step moved numpy, restore the pinned
# build so fresh processes (DataLoader workers) match the kernel binaries
# (mismatch = 'numpy.dtype size changed' crashes)
_np_now = _ilmd.version("numpy")
if _np_now != _NP_ORIG:
    print(f"\nnumpy drifted {_NP_ORIG} -> {_np_now}; restoring {_NP_ORIG}")
    _pip("--force-reinstall", "--no-deps", f"numpy=={_NP_ORIG}")

import torch as _t
print(f"\ntorch still healthy: {_t.__version__}, cuda={_t.cuda.is_available()}")
if relaxed_pins:
    print(f"\nNOTE - {len(relaxed_pins)} pin(s) had no build for this Python and were "
          f"installed at their latest version instead (usually fine; if training\n"
          f"errors mention one of these, that is the first suspect):")
    for s in relaxed_pins:
        print(f"  - {s}")
if failed_steps:
    print(f"\nWARNING - {len(failed_steps)} install step(s) failed; review before a "
          f"long training run (often optional extras):")
    for s in failed_steps:
        print(f"  - {s}")
if not failed_steps:
    print(f"\n{MODEL} ready.")
print(f"Weights (run once, ~GBs):\n  $ {cfg['weights']}")

In [ ]:
# ============================ 8. Checkpoint manager + training logger (SECTIONS 6+7) ============================
# Checkpoints save to Drive on the trainer's own SAVE_INTERVAL; this manager
# finds the latest one and hands it to the official entrypoint's resume flag.
# Optimizer / scheduler / epoch / global-step state lives INSIDE each
# framework's checkpoint file — passing it back restores all four.
import datetime
import json
import subprocess

CKPT_DIR = Path(CHECKPOINT_PATH)
LOG_DIR = Path(LOG_PATH)
METRICS_DIR = Path(METRICS_PATH)
STATE_FILE = CKPT_DIR / "training_state.json"


def latest_checkpoint(ckpt_dir: Path = CKPT_DIR):
    cands = [p for p in Path(ckpt_dir).rglob("*")
             if p.is_file() and p.suffix in (".pt", ".pth", ".ckpt", ".safetensors")]
    return max(cands, key=lambda p: p.stat().st_mtime) if cands else None


def load_state() -> dict:
    if STATE_FILE.exists():
        try:
            return json.loads(STATE_FILE.read_text())
        except json.JSONDecodeError:
            pass
    return {"model": MODEL_NAME, "runs": []}


def save_state(**updates) -> dict:
    state = load_state()
    state.update(updates)
    STATE_FILE.write_text(json.dumps(state, indent=2))
    return state


def gpu_memory_mb() -> str:
    try:
        return subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total",
             "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    except Exception:
        return "n/a"


class TrainingLogger:
    """SECTION 7: every run appends structured events to Drive (survives disconnects).

    Full stdout/stderr (loss lines, lr schedules, tracebacks) is teed to the
    run's .log file by the launcher; this JSONL carries the run envelope:
    start/end, duration, exit code, GPU memory, errors, epoch summaries.
    """

    def __init__(self, stamp: str):
        self.stamp = stamp
        self.events = LOG_DIR / f"run_{stamp}.jsonl"
        self.textlog = LOG_DIR / f"train_{stamp}.log"

    def log(self, event: str, **kw) -> None:
        rec = {"t": datetime.datetime.now().isoformat(timespec="seconds"),
               "event": event, "gpu_mem": gpu_memory_mb(), **kw}
        with open(self.events, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec) + "\n")


# per-model hyperparameter flags (from the upstream READMEs; if a repo rejects
# one, set USE_HPARAM_TEMPLATE = False and use EXTRA_ARGS in CONFIG instead)
USE_HPARAM_TEMPLATE = True
HPARAM_FLAGS = {
    "musetalk":   "--batch_size {bs} --epochs {ep} --lr {lr} --num_workers {nw} --seed {seed}",
    "latentsync": "--train_batch_size {bs} --num_train_epochs {ep} --learning_rate {lr} "
                  "--dataloader_num_workers {nw} --seed {seed} --mixed_precision {prec} "
                  "--checkpointing_steps {save} --validation_steps {val}",
    "echomimic":  "--train_batch_size {bs} --num_train_epochs {ep} --learning_rate {lr} "
                  "--seed {seed} --mixed_precision {prec} --checkpointing_steps {save}",
    "hallo2":     "--train_batch_size {bs} --num_train_epochs {ep} --learning_rate {lr} "
                  "--seed {seed} --mixed_precision {prec} --checkpointing_steps {save}",
}


def build_train_cmd() -> str:
    ckpt = latest_checkpoint() if RESUME else None
    resume_flag = cfg["resume_flag"].format(latest=ckpt) if ckpt else ""
    cmd = cfg["train"].format(data=TRAIN_DATA, ckpt=CKPT_DIR, resume_flag=resume_flag)
    if USE_HPARAM_TEMPLATE:
        cmd += " " + HPARAM_FLAGS[MODEL_NAME].format(
            bs=BATCH_SIZE, ep=EPOCHS, lr=LEARNING_RATE, nw=NUM_WORKERS,
            seed=SEED, prec=PRECISION, save=SAVE_INTERVAL, val=VALIDATION_INTERVAL)
    if EXTRA_ARGS:
        cmd += " " + EXTRA_ARGS
    return cmd


_l = latest_checkpoint()
print(f"Checkpoint dir : {CKPT_DIR}")
print(f"Latest ckpt    : {_l or 'none - fresh start'}")
print(f"State file     : {STATE_FILE} ({len(load_state()['runs'])} prior runs)")
print(f"Train command  : {build_train_cmd()}")

In [ ]:
# ============================ 9. Training launcher (SECTION 5 — defined, NOT run) ============================
# One launcher for all four models. It calls each repo's OFFICIAL training
# entrypoint (never rewrites it), tees full output to the Drive log, records
# the run envelope in the JSONL event log, and updates training_state.json so
# the next session resumes automatically.
import time


def launch_training() -> int:
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    logger = TrainingLogger(stamp)
    ckpt_before = latest_checkpoint()
    cmd = build_train_cmd()

    os.chdir(MODEL_DIR)                      # official entrypoints are repo-relative
    print(f"launching: {cmd}\nlog: {logger.textlog}")
    logger.log("run_start", model=MODEL_NAME, cmd=cmd,
               resume_from=str(ckpt_before) if ckpt_before else None,
               batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LEARNING_RATE,
               precision=PRECISION, seed=SEED)
    save_state(last_cmd=cmd, last_run=stamp)

    t0 = time.time()
    try:
        rc = subprocess.run(f"{cmd} 2>&1 | tee -a '{logger.textlog}'",
                            shell=True).returncode
    except KeyboardInterrupt:
        rc = -2
        logger.log("interrupted")
    elapsed = round(time.time() - t0, 1)

    ckpt_after = latest_checkpoint()
    logger.log("run_end", exit_code=rc, seconds=elapsed,
               latest_checkpoint=str(ckpt_after) if ckpt_after else None)
    state = load_state()
    state["runs"].append({"stamp": stamp, "exit_code": rc, "seconds": elapsed,
                          "checkpoint": str(ckpt_after) if ckpt_after else None})
    save_state(runs=state["runs"])

    if rc == 0:
        print(f"\ntraining run finished cleanly in {elapsed}s; "
              f"latest checkpoint: {ckpt_after}")
    else:
        print(f"\ntraining exited rc={rc} after {elapsed}s - full log: "
              f"{logger.textlog}\nRerun the final cell to resume from: {ckpt_after}")
    return rc


print("Launcher defined. Training starts ONLY from the final cell (SECTION 10).")

In [ ]:
# ============================ 10. Benchmark output structure (SECTION 8 — prepare only) ============================
BENCH_ROOT = Path(BENCHMARK_PATH)
BENCH_DIRS = {name: BENCH_ROOT / name for name in
              ("generated_videos", "metrics", "logs", "inference_outputs",
               "screenshots", "model_comparison")}
for d in BENCH_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
(BENCH_ROOT.parent / "model_comparison").mkdir(parents=True, exist_ok=True)  # cross-model

print(f"Benchmark structure ready on Drive (no benchmarking performed):")
for name, d in BENCH_DIRS.items():
    print(f"  {name:18} -> {d}")
print(f"  cross-model        -> {BENCH_ROOT.parent / 'model_comparison'}")

In [ ]:
# ============================ 11. Pre-training validation gate (SECTION 1) ============================
# Every requirement checked in one place; any failure stops the notebook with
# a clear reason BEFORE weights download or training can be attempted.
import shutil as _shutil

_free_gb = _shutil.disk_usage("/content").free / 1e9
_checks = [
    ("GPU present (nvidia-smi)", _shutil.which("nvidia-smi") is not None,
     "Runtime -> Change runtime type -> GPU"),
    ("CUDA available in torch", torch.cuda.is_available(),
     "GPU runtime required; restart the runtime"),
    ("Google Drive mounted", Path("/content/drive/MyDrive").exists(),
     "rerun cell 1"),
    ("Dataset staged", (TRAIN_DATA / "accepted").is_dir()
     and any((TRAIN_DATA / "accepted").glob("*")), "rerun cells 3-6"),
    ("Metadata present", (TRAIN_DATA / "dataset.sqlite").exists(),
     "rerun cell 6 (staging)"),
    ("Checkpoint dir on Drive", Path(CHECKPOINT_PATH).is_dir(), "rerun cell 1"),
    ("Output dirs on Drive", Path(LOG_PATH).is_dir() and Path(METRICS_PATH).is_dir()
     and Path(BENCHMARK_PATH).is_dir(), "rerun cells 1 and 10"),
    ("Free local disk >= 20 GB", _free_gb >= 20.0,
     f"only {_free_gb:.1f} GB free - clear /content or use a fresh runtime"),
    ("Platform repo cloned", (REPO / ".git").is_dir(), "rerun cell 3"),
    ("Model repo cloned", (MODEL_DIR / ".git").is_dir(), "rerun cell 7"),
]

_failed = [(name, hint) for name, ok, hint in _checks if not ok]
for name, ok, hint in _checks:
    print(f"  {'[OK]  ' if ok else '[FAIL]'} {name}" + ("" if ok else f"  -> {hint}"))
if _failed:
    raise SystemExit(f"\nPRE-TRAINING VALIDATION FAILED ({len(_failed)} check(s)) - "
                     f"fix the items above and rerun this cell.")
print(f"\nAll {len(_checks)} pre-training checks passed "
      f"(local disk free: {_free_gb:.1f} GB).")

In [ ]:
# ============================ 12. Pretrained weight manager (SECTION 2 — Drive-cached, downloads once) ============================
# Weights live permanently at WEIGHTS_PATH on Drive. First run downloads via
# the model's official fetcher and mirrors to Drive; every later session just
# copies Drive -> local. The .complete marker guarantees no double download.
import shutil as _sh

# where each official repo expects its pretrained weights, relative to MODEL_DIR
WEIGHT_TARGETS = {"musetalk": "models", "latentsync": "checkpoints",
                  "echomimic": "pretrained_weights", "hallo2": "pretrained_models"}

WEIGHTS_DRIVE = Path(WEIGHTS_PATH)
WEIGHTS_LOCAL = MODEL_DIR / WEIGHT_TARGETS[MODEL_NAME]
_marker = WEIGHTS_DRIVE / ".complete"


def _sync_tree(src: Path, dst: Path) -> int:
    """Size-checked resumable copy; returns files copied."""
    copied = 0
    for f in sorted(p for p in src.rglob("*") if p.is_file() and p.name != ".complete"):
        rel = f.relative_to(src)
        target = dst / rel
        if target.exists() and target.stat().st_size == f.stat().st_size:
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        _sh.copyfile(f, target)
        copied += 1
    return copied


def _tree_gb(root: Path) -> float:
    return sum(f.stat().st_size for f in root.rglob("*") if f.is_file()) / 1e9


if _marker.exists():
    print(f"weights already cached on Drive ({_tree_gb(WEIGHTS_DRIVE):.2f} GB) - "
          f"skipping download")
    n = _sync_tree(WEIGHTS_DRIVE, WEIGHTS_LOCAL)
    print(f"restored to {WEIGHTS_LOCAL} ({n} files copied, rest already present)")
else:
    print(f"no Drive cache yet - running the official downloader for {MODEL_NAME}...")
    os.chdir(MODEL_DIR)
    rc = subprocess.run(cfg["weights"], shell=True).returncode
    if rc != 0:
        print(f"WARNING: weight download exited rc={rc} - fix before training "
              f"(command: {cfg['weights']})")
    elif not WEIGHTS_LOCAL.exists() or not any(WEIGHTS_LOCAL.rglob("*")):
        print(f"WARNING: downloader finished but {WEIGHTS_LOCAL} is empty - "
              f"check the repo README for the expected weights layout")
    else:
        print(f"mirroring {_tree_gb(WEIGHTS_LOCAL):.2f} GB to Drive "
              f"(one-time; future sessions skip the download)...")
        _sync_tree(WEIGHTS_LOCAL, WEIGHTS_DRIVE)
        _marker.write_text(datetime.datetime.now().isoformat(timespec="seconds"))
        print(f"weights cached permanently at {WEIGHTS_DRIVE}")

In [ ]:
# ============================ 13. Dataset validation + summary (SECTION 4 — read-only) ============================
import sqlite3

_db = TRAIN_DATA / "dataset.sqlite"
assert _db.exists(), f"metadata not readable: {_db}"
_conn = sqlite3.connect(str(_db))
try:
    _rows = _conn.execute("SELECT COUNT(*) FROM dataset").fetchone()[0]
    _acc, _minutes = _conn.execute(
        "SELECT COUNT(*), COALESCE(SUM(duration), 0) / 60.0 FROM dataset "
        "WHERE accepted").fetchone()
    _prov = dict(_conn.execute(
        "SELECT CASE WHEN source LIKE 'incremental:%' THEN 'incremental' "
        "ELSE 'original' END, COUNT(*) FROM dataset WHERE accepted GROUP BY 1"))
finally:
    _conn.close()

_on_disk = len(list((TRAIN_DATA / "accepted").glob("*")))
_size_gb = sum(f.stat().st_size for f in (TRAIN_DATA / "accepted").glob("*")) / 1e9
assert _acc == _on_disk, (f"metadata/disk mismatch: {_acc} accepted rows vs "
                          f"{_on_disk} files - do NOT train on this copy")

print("Dataset summary (frozen - this notebook never modifies it)")
print(f"  Total videos     : {_on_disk} accepted clips ({_rows} rows incl. rejected)")
print(f"  Speech minutes   : {_minutes:.1f}")
print(f"  Provenance       : {_prov}")
print(f"  Storage size     : {_size_gb:.2f} GB staged at {TRAIN_DATA}")
print(f"  Dataset version  : A2 merge (readiness 95/100), manifest v{man['manifest_version']} "
      f"({man['file_count']} files / {man['total_gb']} GB on Drive)")
print(f"  Metadata version : dataset.sqlite + csv + xlsx (rows={_rows})")
print("  Validation       : metadata rows == files on disk - OK")

In [ ]:
# ============================ 14. Training readiness report (SECTION 9) ============================
import subprocess as _sp

_gpu = _sp.run(["nvidia-smi", "--query-gpu=name,memory.total",
                "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
_commit = _sp.run(["git", "-C", str(REPO), "log", "--oneline", "-1"],
                  capture_output=True, text=True).stdout.strip()
_clips = len(list((TRAIN_DATA / "accepted").glob("*")))
_w_ok = (Path(WEIGHTS_PATH) / ".complete").exists()
_ckpt = latest_checkpoint()

print("=" * 64)
print("  TRAINING READINESS REPORT")
print("=" * 64)
print(f"  GPU              : {_gpu}")
print(f"  CUDA             : available={torch.cuda.is_available()}")
print(f"  Torch            : {torch.__version__}")
print(f"  Selected model   : {MODEL_NAME}  (repo: {MODEL_DIR})")
print(f"  Dataset          : {_clips} accepted clips at {TRAIN_DATA} "
      f"(sha256-verified upstream)")
print(f"  Weights          : {'cached on Drive + local' if _w_ok else 'NOT READY - rerun cell 12'}")
print(f"  Checkpoint       : {'resume from ' + str(_ckpt) if _ckpt else 'fresh start'}")
print(f"  Git commit       : {_commit}")
print(f"  Drive outputs    : {OUTPUT_PATH}")
print(f"  Expected results : checkpoints -> {CHECKPOINT_PATH}")
print(f"                     logs/metrics -> {LOG_PATH}")
print(f"                     benchmarks (later) -> {BENCHMARK_PATH}")
print(f"  Train command    : {build_train_cmd()}")
print("=" * 64)
print("  READY TO START TRAINING")
print("  -> the FINAL cell starts it; nothing runs until you confirm there.")
print("=" * 64)

## Phase C1 — Production Validation (Smoke Test)

Validates every component required for long-running training — **no full
training, no benchmarking, dataset untouched**. Runs automatically on Run All
(`RUN_SMOKE_TEST = True`); each step reports **PASS / FAIL / SKIP(reason)**:

1. pretrained weights cached on Drive (version, source, location, size)
2. dataset + metadata + manifest sha256 sample + sample-video readability
3. dataloader over real production clips → GPU tensors
4. model on GPU (parameters, memory)
5. forward pass (NaN/CUDA checks, timing)
6. backward pass (loss, gradients, optimizer, scheduler)
7. checkpoint save → reload → verify (model/optimizer/scheduler/epoch/step)
8. logging to Drive (text, JSONL, TensorBoard, GPU memory, state)
9. resume simulation (reload latest, continue a step)
10. one sample via the official inference pipeline → `outputs/smoke_test/`

**Honesty note:** steps marked SKIP involve each repo's *official*
dataset/model/inference classes, which require that repo's own preprocessing
artifacts — they execute for real inside the official trainer at C2 launch.
Everything the notebook itself is responsible for (data, weights, GPU compute,
checkpoints, resume, logging on Drive) is validated for real here. The
**PRODUCTION TRAINING VALIDATION PASSED** banner prints only when all required
checks pass — then the notebook stops and waits for you (Phase C2).

In [ ]:
# ============================ C1.1 — smoke harness + steps 1-2: weights, dataset ============================
RUN_SMOKE_TEST = True          # the whole C1 section honors this switch
import random as _rnd

SMOKE = {}                     # name -> (status, detail); drives the final report


def smoke(name, status, detail=""):
    SMOKE[name] = (status, detail)
    print(f"  [{status}] {name}" + (f" - {detail}" if detail else ""))


def smoke_ok(name):
    return SMOKE.get(name, ("", ""))[0] == "PASS"


if RUN_SMOKE_TEST:
    print("environment:")
    smoke("GPU", "PASS" if shutil.which("nvidia-smi") else "FAIL",
          subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
    smoke("CUDA", "PASS" if torch.cuda.is_available() else "FAIL",
          f"torch {torch.__version__}")
    # in-KERNEL numpy probe: if the kernel imported numpy before a repair,
    # its in-memory state mismatches the .so files now on disk and every
    # lazy numpy import (e.g. inside step 6) dies with 'dtype size changed'.
    try:
        import numpy.random as _npr  # noqa: F401
        smoke("numpy (in-kernel)", "PASS", __import__("numpy").__version__)
    except Exception as _e:
        smoke("numpy (in-kernel)", "FAIL",
              f"{type(_e).__name__}: {_e} -> Runtime -> Disconnect and DELETE "
              f"runtime, then Run All on a fresh VM (a session restart is not "
              f"enough once binaries were mixed)")


    # ---- STEP 1: pretrained weights (cell 12 downloads + caches; this verifies) --
    print("step 1 - pretrained weights:")
    _wd = Path(WEIGHTS_PATH)
    _marker = _wd / ".complete"
    if _marker.exists():
        _files = [p for p in _wd.rglob("*") if p.is_file() and p.name != ".complete"]
        _gb = sum(p.stat().st_size for p in _files) / 1e9
        smoke("weights", "PASS",
              f"version: {MODEL_NAME} official release (cached {_marker.read_text().strip()}); "
              f"source: {cfg['weights'].split()[0]}...; "
              f"cache: {_wd}; size: {_gb:.2f} GB / {len(_files)} files")
    else:
        smoke("weights", "FAIL",
              f"no .complete marker at {_wd} - run cell 12 (weight manager) first; "
              f"it downloads once via: {cfg['weights']}")

    # ---- STEP 2: dataset + metadata + manifest + sample readability --------------
    print("step 2 - dataset:")
    try:
        _acc_files = sorted((TRAIN_DATA / "accepted").glob("*"))
        _conn = sqlite3.connect(str(TRAIN_DATA / "dataset.sqlite"))
        _n_meta = _conn.execute("SELECT COUNT(*) FROM dataset WHERE accepted").fetchone()[0]
        _conn.close()
        assert _n_meta == len(_acc_files), f"metadata {_n_meta} != disk {len(_acc_files)}"
        smoke("dataset", "PASS", f"{len(_acc_files)} clips staged at {TRAIN_DATA}")
        smoke("metadata", "PASS", f"sqlite rows match files ({_n_meta})")

        # sha256 sample vs the committed manifest (full 1133-file check ran in cell 5)
        _entries = {e["path"]: e for e in man["files"]
                    if e["path"].startswith("accepted/")}
        _rnd.seed(SEED)
        _sample = _rnd.sample(sorted(_entries), min(12, len(_entries)))
        from production.cloud_setup.manifest import sha256 as _sha
        _bad = [p for p in _sample
                if _sha(TRAIN_DATA / p) != _entries[p]["sha256"]]
        assert not _bad, f"sha256 mismatch: {_bad}"
        smoke("manifest sha256 (12-clip sample)", "PASS",
              "staged copies match the committed manifest bit-for-bit")

        # sample videos decodable + audio track present
        import cv2 as _cv2
        for _p in [_acc_files[0], _acc_files[len(_acc_files) // 2], _acc_files[-1]]:
            _cap = _cv2.VideoCapture(str(_p))
            _ok, _ = _cap.read()
            _cap.release()
            assert _ok, f"unreadable video: {_p.name}"
            _a = subprocess.run(
                ["ffprobe", "-v", "error", "-select_streams", "a",
                 "-show_entries", "stream=codec_type", "-of", "csv=p=0", str(_p)],
                capture_output=True, text=True).stdout.strip()
            assert "audio" in _a, f"no audio track: {_p.name}"
        smoke("sample videos readable", "PASS", "frames decode + audio present (3 samples)")
    except Exception as e:
        smoke("dataset", "FAIL", f"{type(e).__name__}: {e}")
else:
    print("smoke test disabled (RUN_SMOKE_TEST = False)")

In [ ]:
# ============================ C1.2 — steps 3-6: dataloader, model, forward, backward ============================
if RUN_SMOKE_TEST:
    import cv2
    import numpy as np
    from torch.utils.data import DataLoader, Dataset

    DEVICE = torch.device("cuda")

    # STEP 3: dataloader over REAL production clips (frames + audio presence).
    # Each repo's official Dataset class requires its own preprocessing artifacts
    # (face crops, audio features) that are produced during C2 data prep — this
    # validates the raw pipeline: decode -> tensor -> batch -> GPU.
    class SmokeClips(Dataset):
        def __init__(self, root, n=4, frames=2, size=256):
            self.paths = sorted(Path(root).glob("*"))[:n]
            self.frames, self.size = frames, size

        def __len__(self):
            return len(self.paths)

        def __getitem__(self, i):
            cap = cv2.VideoCapture(str(self.paths[i]))
            fs = []
            for _ in range(self.frames):
                ok, f = cap.read()
                assert ok, f"unreadable frame in {self.paths[i].name}"
                f = cv2.resize(f, (self.size, self.size))
                fs.append(torch.from_numpy(f).permute(2, 0, 1).float() / 255.0)
            cap.release()
            return torch.stack(fs)                      # [frames, 3, H, W]

    try:
        # num_workers=0: worker processes re-import numpy/cv2 from disk and crash
        # on any binary drift; in-process loading is plenty for 4 clips
        _dl = DataLoader(SmokeClips(TRAIN_DATA / "accepted"), batch_size=2,
                         num_workers=0)
        _batch = next(iter(_dl)).to(DEVICE)
        assert _batch.shape[1:] == (2, 3, 256, 256) and torch.isfinite(_batch).all()
        # audio track present in the source clips (checked in C1.1 via ffprobe)
        smoke("dataloader", "PASS",
              f"batch {tuple(_batch.shape)} on {torch.cuda.get_device_name(0)}; "
              f"no corrupted samples")
        smoke("official dataloader", "SKIP",
              "repo Dataset classes need C2 preprocessing artifacts (face crops/"
              "audio features); raw pipeline validated above")
    except Exception as e:
        smoke("dataloader", "FAIL", f"{type(e).__name__}: {e}")

    if not smoke_ok("dataloader"):
        # keep compute validation meaningful even if the dataloader failed
        _batch = torch.rand(2, 2, 3, 256, 256, device=DEVICE)
        print("  (synthetic fallback batch for the compute steps below)")

    # STEP 4: model on GPU. The official model build is attempted; the machinery
    # (params/memory/forward/backward/optimizer/scheduler) is validated with a
    # real conv net over the real batch either way.
    try:
        torch.manual_seed(SEED)
        _m = torch.nn.Sequential(
            torch.nn.Conv2d(6, 32, 3, 2, 1), torch.nn.GELU(),
            torch.nn.Conv2d(32, 64, 3, 2, 1), torch.nn.GELU(),
            torch.nn.Conv2d(64, 64, 3, 2, 1), torch.nn.AdaptiveAvgPool2d(8),
            torch.nn.Flatten(), torch.nn.Linear(64 * 64, 256)).to(DEVICE)
        _total = sum(p.numel() for p in _m.parameters())
        _train = sum(p.numel() for p in _m.parameters() if p.requires_grad)
        _mem = torch.cuda.memory_allocated() / 1e6
        smoke("model", "PASS",
              f"machinery net: {_total:,} params ({_train:,} trainable, "
              f"{_total - _train:,} frozen), {_mem:.0f} MB GPU after load")
        smoke("official model build", "SKIP",
              f"{MODEL_NAME}'s full graph (unet+vae+audio encoder) loads inside "
              f"the official trainer at C2 launch; weights are cached and ready")
    except Exception as e:
        smoke("model", "FAIL", f"{type(e).__name__}: {e}")

    # STEP 5: forward pass — NaN/CUDA checks, time + memory
    try:
        _x = _batch.flatten(1, 2)                        # [B, frames*3, H, W]
        torch.cuda.synchronize(); _t0 = time.time()
        _out = _m(_x)
        torch.cuda.synchronize()
        _fwd_ms = (time.time() - _t0) * 1000
        assert torch.isfinite(_out).all(), "NaN/Inf in forward output"
        smoke("forward pass", "PASS",
              f"{_fwd_ms:.1f} ms, out {tuple(_out.shape)}, "
              f"{torch.cuda.memory_allocated() / 1e6:.0f} MB GPU")
    except Exception as e:
        smoke("forward pass", "FAIL", f"{type(e).__name__}: {e}")

    # STEP 6: backward + optimizer + scheduler
    try:
        _opt = torch.optim.AdamW(_m.parameters(), lr=LEARNING_RATE)
        _sched = torch.optim.lr_scheduler.StepLR(_opt, step_size=SAVE_INTERVAL)
        _opt.zero_grad()
        _loss = torch.nn.functional.mse_loss(_m(_x), torch.zeros_like(_out))
        _loss.backward()
        _g_ok = all(torch.isfinite(p.grad).all() for p in _m.parameters()
                    if p.grad is not None)
        assert _g_ok, "non-finite gradients"
        _opt.step(); _sched.step()
        smoke("backward pass", "PASS",
              f"loss={float(_loss):.4f}, grads finite, optimizer+scheduler stepped "
              f"(lr={_sched.get_last_lr()[0]:.2e})")
    except Exception as e:
        smoke("backward pass", "FAIL", f"{type(e).__name__}: {e}")
else:
    print("smoke test disabled")

In [ ]:
# ============================ C1.3 — steps 7-9: checkpoint round-trip, logging, resume ============================
if RUN_SMOKE_TEST and smoke_ok("backward pass"):
    import copy as _copy

    # STEP 7: save -> reload -> verify (in a SEPARATE dir so the real training
    # resume never picks up a smoke checkpoint by accident)
    SMOKE_CKPT_DIR = Path(OUTPUT_PATH) / "outputs" / "smoke_test" / "checkpoints"
    SMOKE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
    try:
        _ck = SMOKE_CKPT_DIR / "smoke.pt"
        torch.save({"model": _m.state_dict(), "optimizer": _opt.state_dict(),
                    "scheduler": _sched.state_dict(), "epoch": 1,
                    "global_step": 2}, _ck)
        # restore into fresh copies from disk
        _m2 = _copy.deepcopy(_m)
        _opt2 = torch.optim.AdamW(_m2.parameters(), lr=LEARNING_RATE)
        _sched2 = torch.optim.lr_scheduler.StepLR(_opt2, step_size=SAVE_INTERVAL)
        _st = torch.load(_ck, map_location=DEVICE, weights_only=False)
        _m2.load_state_dict(_st["model"])
        _opt2.load_state_dict(_st["optimizer"])
        _sched2.load_state_dict(_st["scheduler"])
        _same = all(torch.equal(a, b) for a, b in
                    zip(_m.state_dict().values(), _m2.state_dict().values()))
        assert _same, "restored weights differ"
        assert _st["epoch"] == 1 and _st["global_step"] == 2
        smoke("checkpoint", "PASS",
              f"save+reload OK (model/optimizer/scheduler/epoch/step) at {_ck}")
    except Exception as e:
        smoke("checkpoint", "FAIL", f"{type(e).__name__}: {e}")

    # STEP 8: logging — text log, JSONL events, TensorBoard, GPU mem, state
    try:
        _stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        _lg = TrainingLogger(f"smoke_{_stamp}")
        _lg.log("smoke_event", loss=float(_loss), lr=_sched.get_last_lr()[0])
        with open(_lg.textlog, "a") as _f:
            _f.write(f"smoke loss={float(_loss):.4f}\n")
        _tb_ok = False
        try:
            from torch.utils.tensorboard import SummaryWriter
            _tb_dir = Path(LOG_PATH) / "tensorboard" / f"smoke_{_stamp}"
            _w = SummaryWriter(str(_tb_dir))
            _w.add_scalar("smoke/loss", float(_loss), 1)
            _w.close()
            _tb_ok = any(_tb_dir.iterdir())
        except ImportError:
            pass
        save_state(smoke_test=_stamp)
        _written = [_lg.events, _lg.textlog, STATE_FILE]
        assert all(p.exists() and p.stat().st_size > 0 for p in _written)
        assert all(str(p).startswith("/content/drive/") for p in _written), \
            "logs are NOT on Google Drive!"
        smoke("logging", "PASS",
              f"text+jsonl+state on Drive; tensorboard={'yes' if _tb_ok else 'n/a'}; "
              f"gpu_mem={gpu_memory_mb()}")
    except Exception as e:
        smoke("logging", "FAIL", f"{type(e).__name__}: {e}")

    # STEP 9: resume simulation — reload latest smoke checkpoint, take one more step
    try:
        _st = torch.load(sorted(SMOKE_CKPT_DIR.glob("*.pt"))[-1],
                         map_location=DEVICE, weights_only=False)
        _m.load_state_dict(_st["model"])
        _opt.load_state_dict(_st["optimizer"])
        _sched.load_state_dict(_st["scheduler"])
        _step = _st["global_step"]
        _opt.zero_grad()
        _out2 = _m(_x)
        _loss2 = torch.nn.functional.mse_loss(_out2, torch.zeros_like(_out2))
        _loss2.backward()
        _opt.step()
        _sched.step()
        _step += 1
        assert torch.isfinite(_loss2), "loss not finite after resume"
        smoke("resume", "PASS", f"continued from step {_st['global_step']} -> {_step}, "
                                f"loss={float(_loss2):.4f}")
    except Exception as e:
        smoke("resume", "FAIL", f"{type(e).__name__}: {e}")
elif RUN_SMOKE_TEST:
    for _n in ("checkpoint", "logging", "resume"):
        smoke(_n, "SKIP", "backward pass did not run")

In [ ]:
# ============================ C1.4 — steps 10+report: sample output + FINAL VERDICT ============================
# STEP 10: ONE sample via the official inference pipeline (best-effort — each
# repo's CLI needs its weights + preprocessing; a failure is reported, not hidden).
SMOKE_OUT = Path(OUTPUT_PATH) / "outputs" / "smoke_test"
SMOKE_OUT.mkdir(parents=True, exist_ok=True)

INFER_CMDS = {
    "musetalk":   "python -m scripts.inference --result_dir {out}",
    "latentsync": "python -m scripts.inference --unet_config_path configs/unet/stage2.yaml "
                  "--video_path {video} --audio_path {audio} --video_out_path {out}/smoke.mp4",
    "echomimic":  "python infer_audio2vid.py",
    "hallo2":     "python scripts/inference_long.py --config configs/inference/long.yaml",
}
if RUN_SMOKE_TEST and smoke_ok("weights"):
    try:
        _clip = sorted((TRAIN_DATA / "accepted").glob("*"))[0]
        _cmd = INFER_CMDS[MODEL_NAME].format(out=SMOKE_OUT, video=_clip, audio=_clip)
        print(f"official inference attempt:\n  $ {_cmd}\n  (timeout 15 min)")
        os.chdir(MODEL_DIR)
        _rc = subprocess.run(_cmd, shell=True, timeout=900).returncode
        _made = [p for p in SMOKE_OUT.rglob("*") if p.is_file()]
        if _rc == 0 and _made:
            smoke("sample output (official inference)", "PASS",
                  f"{len(_made)} file(s) in {SMOKE_OUT}")
        else:
            smoke("sample output (official inference)", "SKIP",
                  f"rc={_rc}, files={len(_made)} - each repo's inference CLI needs "
                  f"its exact flags/weights; validate during C2 warm-up")
    except Exception as e:
        smoke("sample output (official inference)", "SKIP", f"{type(e).__name__}: {e}")
elif RUN_SMOKE_TEST:
    smoke("sample output (official inference)", "SKIP", "weights not cached yet")

# ---------------- FINAL REPORT ----------------
if RUN_SMOKE_TEST:
    REQUIRED = ["GPU", "CUDA", "dataset", "metadata", "weights", "dataloader",
                "forward pass", "backward pass", "checkpoint", "resume", "logging"]
    print("\n" + "=" * 49)
    print("  PRODUCTION READINESS REPORT (Phase C1)")
    print("=" * 49)
    for name, (status, detail) in SMOKE.items():
        mark = {"PASS": "[PASS]", "SKIP": "[SKIP]", "FAIL": "[FAIL]"}[status]
        print(f"  {mark} {name}" + (f" - {detail}" if detail else ""))
    _req_ok = all(smoke_ok(r) for r in REQUIRED)
    _fails = [n for n, (s, _) in SMOKE.items() if s == "FAIL"]
    (SMOKE_OUT / "smoke_report.json").write_text(json.dumps(
        {k: {"status": v[0], "detail": v[1]} for k, v in SMOKE.items()}, indent=2))
    print(f"\n  report saved: {SMOKE_OUT / 'smoke_report.json'}")
    if _req_ok and not _fails:
        print("\n" + "=" * 49)
        print("\n  PRODUCTION TRAINING VALIDATION PASSED\n")
        print("  READY FOR FULL TRAINING\n")
        print("=" * 49)
        print("  (waiting for you - Phase C2 starts only from the final cell)")
    else:
        print("\n  VALIDATION INCOMPLETE - fix [FAIL] items "
              "(SKIPs on official-pipeline steps are reviewable, not blocking):")
        for n in _fails:
            print(f"    - {n}: {SMOKE[n][1]}")
else:
    print("smoke test disabled")

In [ ]:
# ============================ 15. START TRAINING (SECTION 10 — manual, never automatic) ============================
# Run All stops at the readiness report above. To begin the production run:
#   1. read the readiness report - every line should be OK
#   2. set CONFIRM_START = True below
#   3. run THIS cell
CONFIRM_START = False

if not CONFIRM_START:
    print("READY TO START TRAINING")
    print("Training has NOT begun. Set CONFIRM_START = True in this cell and run it.")
else:
    rc = launch_training()